# CS2229 - Stage 2: rho ablation của SAM trên CIFAR-10

Notebook này là pipeline của **stage 2** (mở rộng 1 trong kế hoạch thực nghiệm): chạy SAM-SGD với các bán kính `rho` khác `0.05` để đo độ nhạy của SAM theo `rho`.

Notebook được nhân bản từ `sam_reproduction_vi.ipynb` (`sam_reproduction_vi_v5`). Data split, augmentation, model, training loop, SAM, cách chọn checkpoint và artifact giữ nguyên. Điểm khác duy nhất: mỗi lần **Run all** chạy SAM với từng giá trị trong `RHO_VALUES` thay vì cặp SGD/SAM. Nhờ vậy kết quả đặt được cạnh `core_sgd_s<SEED>` (tương ứng `rho = 0`) và `core_sam_s<SEED>` (`rho = 0.05`) của cùng seed.

Mục tiêu chính:

- Dùng cùng recipe với core run, chỉ thay `rho`.
- Chọn checkpoint tốt nhất chỉ bằng validation accuracy.
- Lưu đầy đủ config, lịch sử huấn luyện, checkpoint, biểu đồ và thông tin môi trường.
- Tự resume từ checkpoint trên Google Drive khi Colab ngắt phiên.

## Cách chạy trên Google Colab

1. Dùng lại thư mục `MyDrive/CS2229-SAM/` của core run: `data/cifar-10-python.tar.gz` phải có sẵn, `results/` được tự tạo nếu chưa có. Run ID của stage 2 khác core run nên không ghi đè kết quả core.
2. Upload `notebooks/sam_stage2_reproduction_vi.ipynb` của branch được giao lên Colab, chọn **Runtime -> Change runtime type -> GPU**.
3. Chọn **Runtime -> Run all** và cấp quyền Google Drive cho đúng tài khoản chứa thư mục `CS2229-SAM`.

`RUN_PLAN` trong cell cấu hình quyết định **Run all** chạy gì:

| `RUN_PLAN` | Model | Dữ liệu train | Epochs | Run IDs được tạo |
|---|---|---:|---:|---|
| `"smoke"` | WRN-16-1 | 1.024 ảnh | 5 | `smoke_rho_001_s<SEED>`, `smoke_rho_010_s<SEED>`, `smoke_rho_020_s<SEED>` |
| `"rho"` | WRN-28-10 | 45.000 ảnh | 200 | `rho_001_s<SEED>`, `rho_010_s<SEED>`, `rho_020_s<SEED>` |

Branch `main` đặt `RUN_PLAN = "smoke"`. Mỗi branch `run/rho-s<k>` chỉ khác `main` ở cell cấu hình: `RUN_PLAN = "rho"` và `SEED = k`. Kế hoạch hiện tại chỉ có `run/rho-s1`.

## Resume khi Colab hết phiên

Mở lại **đúng notebook đã chạy**, chọn GPU, rồi **Runtime -> Run all**. Notebook bỏ qua run đã hoàn tất, tiếp tục run dở từ epoch kế tiếp và chỉ chạy lại epoch đang dở lúc bị ngắt. Không sửa cell cấu hình và không xóa hay đổi tên thư mục run trên Drive.

Tham chiếu: Foret et al., *Sharpness-Aware Minimization for Efficiently Improving Generalization*, ICLR 2021.

## 1. Kiểm tra môi trường

Google Colab đã có sẵn PyTorch, torchvision, NumPy, pandas và matplotlib. Cell dưới đây chỉ import package, in phiên bản và chọn GPU nếu khả dụng.

Chọn GPU tại **Runtime -> Change runtime type**. `RUN_PLAN = "smoke"` vẫn chạy được bằng CPU nhưng chậm hơn; `RUN_PLAN = "rho"` bắt buộc có GPU.

In [ ]:
# Import toàn bộ dependency dùng trong notebook.
import json
import gc
import io
import os
import pickle
import platform
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from IPython.display import display
from torch.optim import SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.datasets import CIFAR10
from torchvision.datasets.utils import check_integrity, extract_archive


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Thiết bị: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Cảnh báo: chưa có GPU. RUN_PLAN=\"rho\" sẽ bị từ chối.")

## 2. Cấu hình thực nghiệm

Đây là cell duy nhất được phép khác nhau giữa các branch. Khi chạy, mỗi thành viên dùng nguyên notebook của branch được giao và **không sửa** cell này.

- `RUN_PLAN`: `"smoke"` để kiểm tra pipeline nhanh; `"rho"` để chạy SAM 200 epochs với từng giá trị trong `RHO_VALUES` cho seed được giao.
- `SEED`: seed của branch, điều khiển initialization, thứ tự mini-batch, augmentation và label noise.
- `RESUME_IF_EXISTS = True`: tự tiếp tục run chưa hoàn tất có cùng cấu hình và run ID. Run đã có `summary.json` chỉ được kiểm tra lại, không bị ghi đè.
- `RHO_VALUES`: các bán kính SAM của ablation. `rho = 0.05` không có ở đây vì đã chạy trong core run.
- Phần **recipe dùng chung** phải giống core notebook `sam_reproduction_vi_v5` (ghi trong `BASE_NOTEBOOK_VERSION`) và giống nhau trên mọi branch. Mọi thay đổi ảnh hưởng training hoặc evaluation phải được làm trên `main`, tăng `NOTEBOOK_VERSION` và chạy lại các run bị ảnh hưởng.

Khi Run all, notebook yêu cầu quyền Google Drive, đọc dataset từ `MyDrive/CS2229-SAM/data/cifar-10-python.tar.gz` và lưu artifact vào `MyDrive/CS2229-SAM/results`. Nếu thiếu quyền, thiếu dataset hoặc thư mục kết quả không ghi được, notebook dừng trước khi train và không fallback sang `/content`.

In [ ]:
# =========================
# CẤU HÌNH THEO BRANCH
# =========================
RUN_PLAN = "rho"                # "smoke" trên main; "rho" trên branch run/rho-s<SEED>.
SEED = 1                        # Seed được giao cho branch.
RESUME_IF_EXISTS = True         # Tự tiếp tục từ last_checkpoint.pt nếu run chưa hoàn tất.

# =========================
# RECIPE DÙNG CHUNG - KHÔNG SỬA KHI CHẠY
# =========================
METHOD = "sam"                  # Stage 2 chỉ chạy SAM-SGD.
RHO_VALUES = (0.01, 0.10, 0.20) # Thứ tự chạy trong một lần Run all; rho = 0.05 đã có ở core run.
DATASET = "cifar10"             # Notebook hiện chỉ hỗ trợ CIFAR-10.
MODEL = "wrn28x10"              # Giống core matrix.
LABEL_NOISE = 0.0               # Rho ablation không dùng label noise.
EPOCHS = 200                    # Giống core run.

# Hyperparameter dùng chung cho SGD và SAM.
BATCH_SIZE = 128
LEARNING_RATE = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4
NUM_WORKERS = 2
VALIDATION_SIZE = 5_000
SPLIT_SEED = 20261023           # Cố định để mọi method và seed dùng cùng split.

# Dataset được copy từ Drive vào local runtime; artifact bắt buộc lưu trên Google Drive.
DATA_ROOT = Path("/content/data")
GOOGLE_DRIVE_MOUNT_POINT = Path("/content/drive")
GOOGLE_DRIVE_PROJECT_DIR = Path("MyDrive/CS2229-SAM")
GOOGLE_DRIVE_DATA_DIR = GOOGLE_DRIVE_PROJECT_DIR / "data"
GOOGLE_DRIVE_RESULTS_DIR = GOOGLE_DRIVE_PROJECT_DIR / "results"
CIFAR10_ARCHIVE_NAME = "cifar-10-python.tar.gz"
CIFAR10_ARCHIVE_MD5 = "c58f30108f718f92721af3b95e74349a"  # MD5 chính thức, trùng với torchvision.
RESULTS_ROOT = None  # Được gán sau khi mount và kiểm tra quyền ghi thành công.

# Cấu hình nhẹ chỉ dùng khi RUN_PLAN = "smoke".
SMOKE_MODEL = "wrn16x1"
SMOKE_EPOCHS = 5
SMOKE_TRAIN_SIZE = 1_024
SMOKE_VALIDATION_SIZE = 256
SMOKE_TEST_SIZE = 256

# Nhãn phiên bản giúp truy ngược notebook đã tạo artifact.
NOTEBOOK_VERSION = "sam_stage2_reproduction_vi_v1"
# Core notebook mà training pipeline của notebook này được nhân bản từ đó.
BASE_NOTEBOOK_VERSION = "sam_reproduction_vi_v5"

In [ ]:
# Kiểm tra sớm các giá trị dễ nhập sai trước khi mount Drive và tải dữ liệu.
RUN_PLAN = RUN_PLAN.lower()
METHOD = METHOD.lower()
DATASET = DATASET.lower()
MODEL = MODEL.lower()

if RUN_PLAN not in {"smoke", "rho"}:
    raise ValueError("RUN_PLAN phải là 'smoke' hoặc 'rho'.")
if isinstance(SEED, bool) or not isinstance(SEED, int) or SEED < 0:
    raise ValueError("SEED phải là số nguyên không âm.")
if METHOD != "sam":
    raise ValueError("Stage 2 chỉ hỗ trợ METHOD='sam'.")
if not RHO_VALUES or any(
    isinstance(rho, bool) or not isinstance(rho, (int, float)) or rho <= 0
    for rho in RHO_VALUES
):
    raise ValueError("RHO_VALUES phải là dãy khác rỗng gồm các số lớn hơn 0.")
# Run ID mã hóa rho theo phần trăm (0.01 -> 001), nên rho phải là bội của 0.01 và không lặp lại.
RHO_TAGS = [round(rho * 100) for rho in RHO_VALUES]
if any(abs(rho * 100 - tag) > 1e-9 for rho, tag in zip(RHO_VALUES, RHO_TAGS)):
    raise ValueError("Mỗi giá trị trong RHO_VALUES phải là bội của 0.01.")
if len(set(RHO_TAGS)) != len(RHO_TAGS):
    raise ValueError("RHO_VALUES không được lặp lại.")
if DATASET != "cifar10":
    raise ValueError("Notebook hiện chỉ hỗ trợ DATASET='cifar10'.")
if MODEL not in {"wrn28x10", "wrn16x4", "wrn16x1"}:
    raise ValueError("MODEL phải là wrn28x10, wrn16x4 hoặc wrn16x1.")
if not 0.0 <= LABEL_NOISE < 1.0:
    raise ValueError("LABEL_NOISE phải thuộc khoảng [0, 1).")
if RUN_PLAN == "rho" and DEVICE.type != "cuda":
    raise RuntimeError(
        "RUN_PLAN='rho' cần GPU. Chọn Runtime -> Change runtime type -> GPU rồi Run all lại."
    )

SMOKE_TEST = RUN_PLAN == "smoke"
EFFECTIVE_MODEL = SMOKE_MODEL if SMOKE_TEST else MODEL
EFFECTIVE_EPOCHS = SMOKE_EPOCHS if SMOKE_TEST else EPOCHS
RUN_ID_PREFIX = "smoke_rho" if SMOKE_TEST else "rho"
PLANNED_RUN_IDS = {
    float(rho): f"{RUN_ID_PREFIX}_{tag:03d}_s{SEED}"
    for rho, tag in zip(RHO_VALUES, RHO_TAGS)
}

print("Cấu hình đang dùng:")
print(f"  run_plan={RUN_PLAN}, method={METHOD}, model={EFFECTIVE_MODEL}, seed={SEED}")
print(f"  rho_values={list(PLANNED_RUN_IDS)}, label_noise={LABEL_NOISE}, epochs={EFFECTIVE_EPOCHS}")
print(f"  runs={list(PLANNED_RUN_IDS.values())}")
print(f"  notebook_version={NOTEBOOK_VERSION} (pipeline từ {BASE_NOTEBOOK_VERSION})")


### 2.1. Kết nối Google Drive

Cell dưới đây mở hộp thoại cấp quyền của Google Colab. Mỗi thành viên đăng nhập và tự xác nhận quyền cho Drive của mình.

Pipeline dùng cơ chế **fail-closed**:

- chỉ tiếp tục khi hệ điều hành xác nhận `/content/drive` là mount point và `MyDrive` đã tồn tại;
- tạo được thư mục `MyDrive/CS2229-SAM/results`;
- ghi, đọc và xóa được một file kiểm tra nhỏ;
- không chuyển sang lưu tạm trong `/content` khi có lỗi.

Do đó, khi cell này báo lỗi, cần xử lý kết nối Drive trước khi chạy các cell tiếp theo.

In [ ]:
def setup_google_drive_results(
    mount_point=Path("/content/drive"),
    results_relative_dir=Path("MyDrive/CS2229-SAM/results"),
    drive_module=None,
    mount_checker=None,
):
    """Mount Drive, kiểm tra mount thật và xác minh quyền ghi artifact."""
    if drive_module is None:
        try:
            from google.colab import drive as drive_module
        except ImportError as error:
            raise RuntimeError(
                "Google Drive chỉ được mount trong Google Colab. "
                "Hãy upload notebook lên Colab rồi chạy lại."
            ) from error
    if mount_checker is None:
        import os

        mount_checker = os.path.ismount

    try:
        drive_module.mount(str(mount_point))
    except Exception as error:
        raise RuntimeError(
            "Không mount được Google Drive. Notebook đã dừng và chưa bắt đầu training."
        ) from error

    # Không tự tạo MyDrive: thư mục này phải do DriveFS cung cấp sau khi mount thật.
    drive_root = mount_point / "MyDrive"
    if not mount_checker(str(mount_point)) or not drive_root.is_dir():
        raise RuntimeError(
            "Google Drive chưa sẵn sàng tại mount point. "
            "Notebook đã dừng và chưa bắt đầu training."
        )

    if (
        results_relative_dir.is_absolute()
        or not results_relative_dir.parts
        or results_relative_dir.parts[0] != "MyDrive"
        or ".." in results_relative_dir.parts
    ):
        raise ValueError("GOOGLE_DRIVE_RESULTS_DIR phải là đường dẫn tương đối bên trong MyDrive.")

    results_directory = mount_point / results_relative_dir
    probe_path = None
    try:
        from uuid import uuid4

        results_directory.mkdir(parents=True, exist_ok=True)
        probe_path = results_directory / f".write_probe_{uuid4().hex}"
        with probe_path.open("x", encoding="utf-8") as probe_file:
            probe_file.write("google-drive-write-ok")
        if probe_path.read_text(encoding="utf-8") != "google-drive-write-ok":
            raise OSError("Nội dung file kiểm tra không khớp.")
        probe_path.unlink()
    except Exception as error:
        try:
            if probe_path is not None and probe_path.exists():
                probe_path.unlink()
        except OSError:
            pass
        raise RuntimeError(
            "Google Drive đã mount nhưng thư mục kết quả không ghi được. "
            "Notebook đã dừng và chưa bắt đầu training."
        ) from error

    return results_directory


RESULTS_ROOT = setup_google_drive_results(
    mount_point=GOOGLE_DRIVE_MOUNT_POINT,
    results_relative_dir=GOOGLE_DRIVE_RESULTS_DIR,
)
print(f"Artifact sẽ được lưu tại: {RESULTS_ROOT}")

### 2.2. Chuẩn bị CIFAR-10 từ Google Drive

Dataset được đọc từ `MyDrive/CS2229-SAM/data/cifar-10-python.tar.gz` thay vì tải từ Internet, vì tải trực tiếp trên Colab có thể mất gần một giờ.

Cell dưới đây copy archive vào `/content/data`, kiểm tra MD5 chính thức, giải nén và để torchvision kiểm tra integrity từng batch. Nếu không tìm thấy file hoặc MD5 không khớp, notebook dừng trước khi train. Khi đó, upload lại đúng file tên `cifar-10-python.tar.gz` vào thư mục `data`; Drive có thể tự đổi tên file trùng thành `cifar-10-python (1).tar.gz`.

In [ ]:
def prepare_cifar10_from_drive(drive_archive_path, data_root):
    """Copy CIFAR-10 archive từ Drive vào local runtime, kiểm tra MD5 rồi giải nén."""
    drive_archive_path = Path(drive_archive_path)
    data_root = Path(data_root)
    if not drive_archive_path.is_file():
        raise FileNotFoundError(
            f"Không tìm thấy {drive_archive_path}. Hãy upload {CIFAR10_ARCHIVE_NAME} "
            f"vào {GOOGLE_DRIVE_DATA_DIR} rồi Run all lại."
        )

    data_root.mkdir(parents=True, exist_ok=True)
    local_archive_path = data_root / CIFAR10_ARCHIVE_NAME
    if not check_integrity(str(local_archive_path), CIFAR10_ARCHIVE_MD5):
        temporary_path = local_archive_path.with_name(f"{local_archive_path.name}.tmp")
        shutil.copyfile(drive_archive_path, temporary_path)
        temporary_path.replace(local_archive_path)
        if not check_integrity(str(local_archive_path), CIFAR10_ARCHIVE_MD5):
            local_archive_path.unlink()
            raise RuntimeError(
                f"MD5 của {drive_archive_path} không khớp bản CIFAR-10 chính thức. "
                "File có thể bị hỏng khi upload; hãy upload lại rồi Run all lại."
            )

    extract_archive(str(local_archive_path), str(data_root))
    # torchvision kiểm tra MD5 từng batch và báo lỗi nếu dữ liệu giải nén bị hỏng.
    CIFAR10(data_root, train=True, download=False)
    CIFAR10(data_root, train=False, download=False)
    return local_archive_path


prepare_cifar10_from_drive(
    GOOGLE_DRIVE_MOUNT_POINT / GOOGLE_DRIVE_DATA_DIR / CIFAR10_ARCHIVE_NAME,
    DATA_ROOT,
)
print(f"CIFAR-10 đã sẵn sàng tại: {DATA_ROOT}")

## 3. Reproducibility và utility

Cell này cố định các nguồn ngẫu nhiên chính. `SPLIT_SEED` tách khỏi `SEED` để toàn bộ run dùng cùng train/validation split, còn `SEED` điều khiển initialization, thứ tự mini-batch và augmentation.

PyTorch vẫn có thể có sai khác nhỏ giữa GPU hoặc phiên bản CUDA khác nhau. `environment.txt` sẽ ghi lại môi trường để giải thích các sai khác đó.

In [ ]:
def seed_everything(seed):
    """Cố định các nguồn ngẫu nhiên thường dùng trong pipeline."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Chế độ deterministic ưu tiên khả năng tái lập hơn tốc độ.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    """Đồng bộ seed của NumPy và random trong từng DataLoader worker."""
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


class AverageMeter:
    """Tính trung bình có trọng số cho loss hoặc accuracy."""

    def __init__(self):
        self.total = 0.0
        self.count = 0

    def update(self, value, count):
        self.total += float(value) * count
        self.count += count

    @property
    def average(self):
        return self.total / max(self.count, 1)


seed_everything(SEED)

## 4. Chuẩn bị CIFAR-10

Training augmentation bám theo thiết lập phổ biến của WideResNet trên CIFAR-10:

- padding 4 pixels, random crop 32 x 32;
- random horizontal flip;
- chuẩn hóa theo thống kê CIFAR-10.

Validation và test chỉ chuẩn hóa, không dùng augmentation. Label noise chỉ được áp dụng lên training subset.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD = (0.2470, 0.2435, 0.2616)
CIFAR10_CLASSES = (
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
)


class NoisySubset(Dataset):
    """Subset có thể thay một tỷ lệ nhãn bằng nhãn sai xác định theo seed."""

    def __init__(self, dataset, indices, noise_rate=0.0, noise_seed=0):
        self.dataset = dataset
        self.indices = list(indices)
        self.targets = [int(dataset.targets[index]) for index in self.indices]

        if noise_rate > 0:
            rng = np.random.default_rng(noise_seed)
            noisy_count = int(round(len(self.targets) * noise_rate))
            noisy_positions = rng.choice(len(self.targets), size=noisy_count, replace=False)

            # Cộng offset từ 1 đến 9 để nhãn mới luôn khác nhãn gốc.
            for position in noisy_positions:
                offset = int(rng.integers(1, 10))
                self.targets[position] = (self.targets[position] + offset) % 10

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        image, _ = self.dataset[self.indices[position]]
        return image, self.targets[position]


def make_split_indices(dataset_size, validation_size, split_seed):
    """Tạo train/validation split cố định và dùng lại cho mọi run."""
    generator = torch.Generator().manual_seed(split_seed)
    permutation = torch.randperm(dataset_size, generator=generator).tolist()
    validation_indices = permutation[:validation_size]
    train_indices = permutation[validation_size:]
    return train_indices, validation_indices


def build_dataloaders():
    """Tạo các DataLoader và giữ split giống nhau giữa SGD với SAM."""
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])
    evaluation_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
    ])

    augmented_dataset = CIFAR10(DATA_ROOT, train=True, download=False, transform=train_transform)
    evaluation_dataset = CIFAR10(DATA_ROOT, train=True, download=False, transform=evaluation_transform)
    test_dataset = CIFAR10(DATA_ROOT, train=False, download=False, transform=evaluation_transform)

    train_indices, validation_indices = make_split_indices(
        len(augmented_dataset), VALIDATION_SIZE, SPLIT_SEED
    )

    if SMOKE_TEST:
        train_indices = train_indices[:SMOKE_TRAIN_SIZE]
        validation_indices = validation_indices[:SMOKE_VALIDATION_SIZE]
        test_dataset = Subset(test_dataset, range(SMOKE_TEST_SIZE))

    train_dataset = NoisySubset(
        augmented_dataset,
        train_indices,
        noise_rate=LABEL_NOISE,
        noise_seed=SEED,
    )
    train_evaluation_dataset = Subset(evaluation_dataset, train_indices)
    validation_dataset = Subset(evaluation_dataset, validation_indices)

    loader_generator = torch.Generator().manual_seed(SEED)
    common_loader_options = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE.type == "cuda",
        "worker_init_fn": seed_worker,
        # Tạo lại worker mỗi epoch để RNG có thể phục hồi sau khi runtime restart.
        "persistent_workers": False,
    }

    train_loader = DataLoader(
        train_dataset,
        shuffle=True,
        generator=loader_generator,
        **common_loader_options,
    )
    train_evaluation_loader = DataLoader(
        train_evaluation_dataset,
        shuffle=False,
        **common_loader_options,
    )
    validation_loader = DataLoader(
        validation_dataset,
        shuffle=False,
        **common_loader_options,
    )
    test_loader = DataLoader(
        test_dataset,
        shuffle=False,
        **common_loader_options,
    )

    split_info = {
        "split_seed": SPLIT_SEED,
        "train_size": len(train_dataset),
        "validation_size": len(validation_dataset),
        "test_size": len(test_dataset),
    }
    return train_loader, train_evaluation_loader, validation_loader, test_loader, split_info

## 5. WideResNet cho CIFAR-10

Stage 2 dùng `wrn28x10` giống core matrix. Hai biến thể nhỏ hơn vẫn được giữ:

- `wrn16x4`: fallback chỉ dùng nếu cả nhóm cùng quyết định đổi model.
- `wrn16x1`: chỉ dùng cho smoke test và kiểm tra demo.

Nếu nhóm đổi model chính, tất cả thành viên phải dùng cùng một model, tăng `NOTEBOOK_VERSION` và ghi rõ đây là *reduced reproduction*.

In [ ]:
class WideBasicBlock(nn.Module):
    """Residual block dùng trong WideResNet cho ảnh CIFAR."""

    def __init__(self, in_channels, out_channels, stride, dropout_rate):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.dropout_rate = dropout_rate

        self.shortcut = None
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Conv2d(
                in_channels, out_channels, kernel_size=1, stride=stride, bias=False
            )

    def forward(self, inputs):
        outputs = F.relu(self.bn1(inputs), inplace=True)
        shortcut = inputs if self.shortcut is None else self.shortcut(outputs)
        outputs = self.conv1(outputs)
        outputs = F.relu(self.bn2(outputs), inplace=True)
        if self.dropout_rate > 0:
            outputs = F.dropout(outputs, p=self.dropout_rate, training=self.training)
        outputs = self.conv2(outputs)
        return outputs + shortcut


class WideResNet(nn.Module):
    """WideResNet với depth = 6n + 4 dành cho CIFAR-10."""

    def __init__(self, depth=28, widen_factor=10, dropout_rate=0.0, num_classes=10):
        super().__init__()
        if (depth - 4) % 6 != 0:
            raise ValueError("WideResNet yêu cầu depth có dạng 6n + 4.")

        blocks_per_group = (depth - 4) // 6
        channels = [16, 16 * widen_factor, 32 * widen_factor, 64 * widen_factor]

        self.conv1 = nn.Conv2d(3, channels[0], kernel_size=3, padding=1, bias=False)
        self.group1 = self._make_group(
            blocks_per_group, channels[0], channels[1], stride=1, dropout_rate=dropout_rate
        )
        self.group2 = self._make_group(
            blocks_per_group, channels[1], channels[2], stride=2, dropout_rate=dropout_rate
        )
        self.group3 = self._make_group(
            blocks_per_group, channels[2], channels[3], stride=2, dropout_rate=dropout_rate
        )
        self.bn = nn.BatchNorm2d(channels[3])
        self.classifier = nn.Linear(channels[3], num_classes)
        self._initialize_weights()

    @staticmethod
    def _make_group(block_count, in_channels, out_channels, stride, dropout_rate):
        blocks = []
        for block_index in range(block_count):
            block_stride = stride if block_index == 0 else 1
            block_in_channels = in_channels if block_index == 0 else out_channels
            blocks.append(
                WideBasicBlock(
                    block_in_channels, out_channels, block_stride, dropout_rate
                )
            )
        return nn.Sequential(*blocks)

    def _initialize_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Conv2d):
                nn.init.kaiming_normal_(module.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(module, nn.BatchNorm2d):
                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Linear):
                nn.init.zeros_(module.bias)

    def forward(self, inputs):
        outputs = self.conv1(inputs)
        outputs = self.group1(outputs)
        outputs = self.group2(outputs)
        outputs = self.group3(outputs)
        outputs = F.relu(self.bn(outputs), inplace=True)
        outputs = F.adaptive_avg_pool2d(outputs, output_size=1)
        outputs = torch.flatten(outputs, start_dim=1)
        return self.classifier(outputs)


def build_model(model_name):
    """Khởi tạo đúng biến thể WideResNet từ tên cấu hình."""
    model_specs = {
        "wrn28x10": (28, 10),
        "wrn16x4": (16, 4),
        "wrn16x1": (16, 1),
    }
    if model_name not in model_specs:
        raise ValueError(f"Model chưa được hỗ trợ: {model_name}")
    depth, widen_factor = model_specs[model_name]
    return WideResNet(depth=depth, widen_factor=widen_factor, num_classes=10)

## 6. Sharpness-Aware Minimization

SAM thực hiện hai lần forward/backward trên **cùng mini-batch**:

1. Tính gradient tại trọng số hiện tại và dịch trọng số tới điểm lân cận có loss lớn hơn.
2. Tính gradient tại trọng số đã perturb rồi dùng base optimizer để cập nhật trọng số gốc.

BatchNorm running mean/variance và `num_batches_tracked` chỉ được cập nhật ở pass thứ nhất. Ở pass thứ hai, tracking được tạm tắt nhưng BatchNorm vẫn chuẩn hóa bằng thống kê của mini-batch.

In [ ]:
class SAM(torch.optim.Optimizer):
    """SAM wrapper cho một base optimizer, mặc định dùng SGD."""

    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        if rho <= 0:
            raise ValueError("rho phải lớn hơn 0.")

        defaults = {"rho": rho, "adaptive": adaptive, **kwargs}
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        """Perturb trọng số theo hướng gradient có norm lớn nhất."""
        gradient_norm = self._gradient_norm()
        for group in self.param_groups:
            scale = group["rho"] / (gradient_norm + 1e-12)
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                factor = torch.pow(parameter, 2) if group["adaptive"] else 1.0
                perturbation = factor * parameter.grad * scale.to(parameter)
                parameter.add_(perturbation)
                self.state[parameter]["perturbation"] = perturbation

        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        """Khôi phục trọng số rồi gọi base optimizer để cập nhật thật."""
        for group in self.param_groups:
            for parameter in group["params"]:
                if "perturbation" not in self.state[parameter]:
                    continue
                parameter.sub_(self.state[parameter].pop("perturbation"))

        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    def _gradient_norm(self):
        """Tính L2 norm chung của gradient trên mọi parameter."""
        shared_device = self.param_groups[0]["params"][0].device
        norms = []
        for group in self.param_groups:
            for parameter in group["params"]:
                if parameter.grad is None:
                    continue
                factor = torch.abs(parameter) if group["adaptive"] else 1.0
                norms.append((factor * parameter.grad).norm(p=2).to(shared_device))
        if not norms:
            raise RuntimeError("Không tìm thấy gradient để thực hiện SAM first_step.")
        return torch.norm(torch.stack(norms), p=2)

    def step(self, closure=None):
        raise NotImplementedError("SAM cần gọi first_step và second_step tường minh.")

    def state_dict(self):
        return self.base_optimizer.state_dict()

    def load_state_dict(self, state_dict):
        self.base_optimizer.load_state_dict(state_dict)
        self.param_groups = self.base_optimizer.param_groups


def disable_running_stats(model):
    """Ngừng cập nhật BatchNorm running statistics trong SAM pass thứ hai."""
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            if not hasattr(module, "backup_momentum"):
                module.backup_momentum = module.momentum
                module.backup_track_running_stats = module.track_running_stats
            module.momentum = 0
            module.track_running_stats = False


def enable_running_stats(model):
    """Khôi phục BatchNorm momentum trước SAM pass thứ nhất."""
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm) and hasattr(
            module, "backup_momentum"
        ):
            module.momentum = module.backup_momentum
            module.track_running_stats = module.backup_track_running_stats
            del module.backup_momentum
            del module.backup_track_running_stats


def build_optimizer(model, method, rho):
    """Tạo SGD hoặc SAM-SGD với cùng hyperparameter của base optimizer."""
    optimizer_arguments = {
        "lr": LEARNING_RATE,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": True,
    }
    if method == "sgd":
        return SGD(model.parameters(), **optimizer_arguments)
    if method == "sam":
        return SAM(model.parameters(), SGD, rho=rho, **optimizer_arguments)
    raise ValueError(f"Method chưa được hỗ trợ: {method}")

## 7. Huấn luyện và validation

Hai method dùng chung `train_one_epoch`. Nhánh SGD có một backward pass; nhánh SAM có hai backward pass nhưng không lấy batch mới ở giữa. Validation luôn chạy với `model.eval()` và không cập nhật gradient.

In [ ]:
def batch_accuracy(logits, targets):
    """Trả về số dự đoán đúng trong mini-batch."""
    predictions = logits.argmax(dim=1)
    return int((predictions == targets).sum().item())


def train_one_epoch(model, loader, criterion, optimizer, method, device):
    """Huấn luyện một epoch bằng SGD hoặc SAM trên cùng interface."""
    model.train()
    loss_meter = AverageMeter()
    correct_count = 0
    sample_count = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        enable_running_stats(model)
        optimizer.zero_grad()

        # Pass thứ nhất luôn tồn tại ở cả SGD và SAM.
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()

        if method == "sam":
            optimizer.first_step(zero_grad=True)

            # Dùng lại chính images và targets của pass thứ nhất.
            disable_running_stats(model)
            perturbed_logits = model(images)
            perturbed_loss = criterion(perturbed_logits, targets)
            perturbed_loss.backward()
            optimizer.second_step(zero_grad=True)
            enable_running_stats(model)
        elif method == "sgd":
            optimizer.step()
        else:
            raise ValueError(f"Method chưa được hỗ trợ: {method}")

        batch_size = targets.size(0)
        loss_meter.update(loss.item(), batch_size)
        correct_count += batch_accuracy(logits.detach(), targets)
        sample_count += batch_size

    return {
        "loss": loss_meter.average,
        "accuracy": correct_count / max(sample_count, 1),
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Đánh giá loss và accuracy mà không thay đổi model."""
    model.eval()
    loss_meter = AverageMeter()
    correct_count = 0
    sample_count = 0

    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, targets)

        batch_size = targets.size(0)
        loss_meter.update(loss.item(), batch_size)
        correct_count += batch_accuracy(logits, targets)
        sample_count += batch_size

    return {
        "loss": loss_meter.average,
        "accuracy": correct_count / max(sample_count, 1),
    }

## 8. Artifact, checkpoint và experiment runner

Mỗi run hoàn tất tạo một thư mục `results/<run_id>/` gồm:

```text
config.json
history.csv
summary.json
best_checkpoint.pt
last_checkpoint.pt
learning_curves.png
environment.txt
```

- `best_checkpoint.pt` chỉ cập nhật khi validation accuracy tốt hơn và được dùng cho đánh giá cuối.
- `last_checkpoint.pt` được ghi sau mỗi epoch và chứa model, optimizer, scheduler, lịch sử, thời gian, peak memory cùng RNG state.
- Mỗi checkpoint có thêm một bản `.backup` cố định. Checkpoint được ghi đè tại chỗ thay vì ghi file tạm rồi rename, vì trên Google Drive file bị xóa hoặc bị rename đè sẽ vào Trash và vẫn tính dung lượng. Nếu phiên ngắt khi đang ghi, pipeline tự dùng bản còn đọc được.
- `history.csv` ghi thêm `gpu_name` của phiên đã chạy từng epoch, vì Colab có thể cấp GPU khác nhau sau mỗi lần resume.
- Nếu Colab ngắt phiên, Run all lại cùng notebook; pipeline tiếp tục từ epoch kế tiếp.
- Nếu ngắt giữa một epoch, epoch chưa hoàn thành sẽ chạy lại. Nếu `summary.json` đã tồn tại, run được xem là hoàn tất và không bị ghi đè.


In [ ]:
REQUIRED_ARTIFACTS = {
    "config.json",
    "history.csv",
    "summary.json",
    "best_checkpoint.pt",
    "last_checkpoint.pt",
    "learning_curves.png",
    "environment.txt",
}
REQUIRED_SUMMARY_FIELDS = {
    "run_id",
    "method",
    "dataset",
    "model",
    "seed",
    "rho",
    "label_noise",
    "best_validation_accuracy",
    "test_accuracy",
    "final_train_accuracy",
    "final_train_loss",
    "generalization_gap",
    "total_training_seconds",
    "mean_epoch_seconds",
    "peak_gpu_memory_mb",
    "gpu_name",
    "framework_versions",
    "git_commit",
    "notebook_version",
}
REQUIRED_HISTORY_COLUMNS = {
    "epoch",
    "learning_rate",
    "train_loss",
    "train_accuracy",
    "validation_loss",
    "validation_accuracy",
    "epoch_seconds",
    "gpu_name",
}


def atomic_write_text(content, output_path):
    """Ghi text qua file tạm để không làm hỏng file tốt gần nhất."""
    output_path = Path(output_path)
    temporary_path = output_path.with_name(f"{output_path.name}.tmp")
    try:
        temporary_path.write_text(content, encoding="utf-8")
        temporary_path.replace(output_path)
    except Exception:
        try:
            temporary_path.unlink()
        except FileNotFoundError:
            pass
        raise


def checkpoint_backup_path(checkpoint_path):
    """Đường dẫn bản sao cố định đi kèm mỗi checkpoint."""
    checkpoint_path = Path(checkpoint_path)
    return checkpoint_path.with_name(f"{checkpoint_path.name}.backup")


def save_checkpoint(payload, output_path):
    """Ghi checkpoint bằng cách ghi đè tại chỗ, không xóa hay rename file.

    Trên Google Drive mount, file bị xóa hoặc bị rename đè sẽ vào Trash và vẫn tính
    dung lượng trong 30 ngày. Với checkpoint WRN-28-10 khoảng 280 MB ghi mỗi epoch,
    cách ghi file tạm rồi rename có thể làm đầy Drive giữa chừng. Vì vậy bản `.backup`
    được ghi trước, sau đó file chính mới bị ghi đè. Nếu phiên ngắt khi đang ghi,
    ít nhất một trong hai file vẫn đầy đủ để load_checkpoint() sử dụng.
    """
    output_path = Path(output_path)
    buffer = io.BytesIO()
    torch.save(payload, buffer)
    for path in (checkpoint_backup_path(output_path), output_path):
        with path.open("wb") as checkpoint_file:
            checkpoint_file.write(buffer.getbuffer())
            checkpoint_file.flush()
            os.fsync(checkpoint_file.fileno())


def load_checkpoint(checkpoint_path, map_location="cpu", required=True):
    """Load file chính; nếu file thiếu hoặc bị ghi dở thì dùng bản `.backup`."""
    checkpoint_path = Path(checkpoint_path)
    errors = []
    for path in (checkpoint_path, checkpoint_backup_path(checkpoint_path)):
        try:
            checkpoint = torch.load(path, map_location=map_location, weights_only=False)
        except (OSError, EOFError, RuntimeError, pickle.UnpicklingError) as error:
            errors.append(f"{path.name}: {type(error).__name__}: {error}")
            continue
        if path != checkpoint_path:
            print(f"Cảnh báo: {checkpoint_path.name} không đọc được, dùng {path.name}.")
        return checkpoint

    if not required:
        return None
    raise RuntimeError(
        f"Không load được {checkpoint_path} hoặc bản backup: " + "; ".join(errors)
    )


def capture_random_state(loader_generator):
    """Chụp các RNG state cần thiết để tiếp tục ở ranh giới epoch."""
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        "loader_generator": (
            loader_generator.get_state() if loader_generator is not None else None
        ),
    }


def restore_random_state(state, loader_generator):
    """Khôi phục RNG state sau khi model, optimizer và scheduler đã được tạo."""
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])

    cuda_state = state.get("cuda")
    if cuda_state is not None:
        if not torch.cuda.is_available():
            raise RuntimeError(
                "Checkpoint được tạo bằng CUDA nhưng phiên hiện tại không có GPU. "
                "Hãy đổi Colab runtime sang GPU trước khi resume."
            )
        torch.cuda.set_rng_state_all(cuda_state)

    loader_state = state.get("loader_generator")
    if loader_state is not None:
        if loader_generator is None:
            raise RuntimeError("Checkpoint có DataLoader RNG nhưng loader hiện tại không có generator.")
        loader_generator.set_state(loader_state)


def validate_resume_config(saved_config, current_config):
    """Từ chối resume nếu recipe thực nghiệm đã thay đổi."""
    if not isinstance(saved_config, dict):
        raise ValueError("last_checkpoint.pt không chứa config hợp lệ.")

    all_fields = sorted(set(saved_config) | set(current_config))
    mismatches = {
        field: {
            "checkpoint": saved_config.get(field, "<missing>"),
            "current": current_config.get(field, "<missing>"),
        }
        for field in all_fields
        if saved_config.get(field, "<missing>") != current_config.get(field, "<missing>")
    }
    if mismatches:
        details = ", ".join(
            f"{field}: {values['checkpoint']!r} -> {values['current']!r}"
            for field, values in mismatches.items()
        )
        raise ValueError(
            "Không thể resume vì cấu hình hiện tại khác checkpoint: " + details
        )


def prepare_run_directory(results_root, run_id, config, resume_if_exists=True):
    """Tạo run mới hoặc load checkpoint cuối của một run chưa hoàn tất."""
    results_root.mkdir(parents=True, exist_ok=True)
    run_directory = results_root / run_id
    if not run_directory.exists():
        run_directory.mkdir()
        return run_directory, None

    if (run_directory / "summary.json").exists():
        raise FileExistsError(
            f"Run đã hoàn tất và sẽ không bị ghi đè: {run_directory}. "
            "Hãy đổi RUN_ID cho một run mới."
        )
    if not resume_if_exists:
        raise FileExistsError(
            f"Run directory đã tồn tại: {run_directory}. "
            "Đặt RESUME_IF_EXISTS=True để tiếp tục run chưa hoàn tất."
        )

    latest_checkpoint_path = run_directory / "last_checkpoint.pt"
    if not (
        latest_checkpoint_path.is_file()
        or checkpoint_backup_path(latest_checkpoint_path).is_file()
    ):
        raise FileExistsError(
            f"Run directory đã tồn tại nhưng không có last_checkpoint.pt: {run_directory}. "
            "Không thể xác định trạng thái an toàn để resume; hãy đổi RUN_ID hoặc xử lý "
            "thư mục cũ một cách chủ động."
        )

    checkpoint = load_checkpoint(latest_checkpoint_path)
    validate_resume_config(checkpoint.get("config"), config)
    return run_directory, checkpoint


def get_git_commit():
    """Lấy commit hiện tại nếu notebook đang chạy trong một Git repository."""
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return "unavailable"


def collect_environment():
    """Thu thập thông tin đủ để đối chiếu môi trường giữa các thành viên."""
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {
        "python": sys.version.replace("\n", " "),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "gpu_name": gpu_name,
        "git_commit": get_git_commit(),
        "notebook_version": NOTEBOOK_VERSION,
    }


def save_learning_curves(history, output_path):
    """Vẽ loss và accuracy theo epoch để dùng cho báo cáo."""
    history_frame = pd.DataFrame(history)
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    axes[0].plot(history_frame["epoch"], history_frame["train_loss"], label="Train")
    axes[0].plot(history_frame["epoch"], history_frame["validation_loss"], label="Validation")
    axes[0].set(title="Loss theo epoch", xlabel="Epoch", ylabel="Loss")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].plot(
        history_frame["epoch"], 100 * history_frame["train_accuracy"], label="Train"
    )
    axes[1].plot(
        history_frame["epoch"],
        100 * history_frame["validation_accuracy"],
        label="Validation",
    )
    axes[1].set(title="Accuracy theo epoch", xlabel="Epoch", ylabel="Accuracy (%)")
    axes[1].grid(alpha=0.3)
    axes[1].legend()

    figure.tight_layout()
    figure.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.close(figure)


def make_best_checkpoint(epoch, model, optimizer, validation_accuracy, config):
    """Tạo payload checkpoint dùng để chọn và đánh giá model tốt nhất."""
    return {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "validation_accuracy": validation_accuracy,
        "config": config,
    }


def validate_resume_checkpoint(checkpoint):
    """Kiểm tra các field bắt buộc trước khi nạp trạng thái training."""
    required_fields = {
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "history",
        "best_epoch",
        "best_validation_accuracy",
        "elapsed_training_seconds",
        "peak_gpu_memory_mb",
        "random_state",
        "config",
    }
    missing_fields = required_fields - set(checkpoint)
    if missing_fields:
        raise RuntimeError(
            f"last_checkpoint.pt thiếu field: {sorted(missing_fields)}"
        )

    completed_epoch = int(checkpoint["epoch"])
    history_epochs = [int(row["epoch"]) for row in checkpoint["history"]]
    if history_epochs != list(range(1, completed_epoch + 1)):
        raise RuntimeError(
            "Lịch sử trong last_checkpoint.pt không khớp với epoch đã hoàn thành."
        )


def ensure_best_checkpoint(run_directory, latest_checkpoint, model, optimizer, config):
    """Khôi phục best checkpoint nếu phiên cũ ngắt giữa hai atomic write."""
    best_epoch = int(latest_checkpoint["best_epoch"])
    if best_epoch == 0:
        return

    best_checkpoint_path = run_directory / "best_checkpoint.pt"
    saved_best = load_checkpoint(best_checkpoint_path, required=False)
    if saved_best is not None and int(saved_best.get("epoch", -1)) == best_epoch:
        return

    if best_epoch != int(latest_checkpoint["epoch"]):
        raise RuntimeError(
            "best_checkpoint.pt không khớp last_checkpoint.pt và không thể tự phục hồi."
        )

    save_checkpoint(
        make_best_checkpoint(
            best_epoch,
            model,
            optimizer,
            latest_checkpoint["best_validation_accuracy"],
            config,
        ),
        best_checkpoint_path,
    )


def build_run_config(run_method, rho, selected_run_id, split_info, parameter_count):
    """Tạo config đầy đủ dùng cho cả run mới và kiểm tra resume."""
    return {
        "run_id": selected_run_id,
        "run_plan": RUN_PLAN,
        "method": run_method,
        "dataset": DATASET,
        "model": EFFECTIVE_MODEL,
        "requested_model": MODEL,
        "seed": SEED,
        "split_seed": SPLIT_SEED,
        "validation_split_size": VALIDATION_SIZE,
        "rho": rho if run_method == "sam" else 0.0,
        "label_noise": LABEL_NOISE,
        "epochs": EFFECTIVE_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "nesterov": True,
        "scheduler": "CosineAnnealingLR",
        "num_workers": NUM_WORKERS,
        "augmentation": ["RandomCrop(32, padding=4)", "RandomHorizontalFlip"],
        "normalization_mean": CIFAR10_MEAN,
        "normalization_std": CIFAR10_STD,
        "smoke_test": SMOKE_TEST,
        "parameter_count": parameter_count,
        "notebook_version": NOTEBOOK_VERSION,
        "base_notebook_version": BASE_NOTEBOOK_VERSION,
        **split_info,
    }


def run_experiment(method, run_id, rho):
    """Chạy mới hoặc resume một cấu hình end-to-end và trả về artifact."""
    run_method = method.lower()
    selected_run_id = run_id
    if not selected_run_id.strip():
        raise ValueError("run_id không được để trống.")
    if run_method not in {"sgd", "sam"}:
        raise ValueError("method phải là 'sgd' hoặc 'sam'.")

    seed_everything(SEED)
    train_loader, train_eval_loader, validation_loader, test_loader, split_info = (
        build_dataloaders()
    )
    model = build_model(EFFECTIVE_MODEL).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, run_method, rho)
    scheduler_optimizer = optimizer.base_optimizer if run_method == "sam" else optimizer
    scheduler = CosineAnnealingLR(scheduler_optimizer, T_max=EFFECTIVE_EPOCHS)

    environment = collect_environment()
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    config = build_run_config(
        run_method,
        rho,
        selected_run_id,
        split_info,
        parameter_count,
    )
    run_directory, resume_checkpoint = prepare_run_directory(
        RESULTS_ROOT,
        selected_run_id,
        config,
        resume_if_exists=RESUME_IF_EXISTS,
    )

    if resume_checkpoint is None:
        atomic_write_text(
            json.dumps(config, ensure_ascii=False, indent=2),
            run_directory / "config.json",
        )
        atomic_write_text(
            "\n".join(f"{key}: {value}" for key, value in environment.items()) + "\n",
            run_directory / "environment.txt",
        )

    history = []
    best_epoch = 0
    best_validation_accuracy = -1.0
    elapsed_before_session = 0.0
    peak_gpu_memory_before_session = 0.0
    start_epoch = 1

    if resume_checkpoint is not None:
        validate_resume_checkpoint(resume_checkpoint)
        model.load_state_dict(resume_checkpoint["model_state_dict"])
        optimizer.load_state_dict(resume_checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(resume_checkpoint["scheduler_state_dict"])
        history = list(resume_checkpoint["history"])
        best_epoch = int(resume_checkpoint["best_epoch"])
        best_validation_accuracy = float(
            resume_checkpoint["best_validation_accuracy"]
        )
        elapsed_before_session = float(
            resume_checkpoint["elapsed_training_seconds"]
        )
        peak_gpu_memory_before_session = float(
            resume_checkpoint["peak_gpu_memory_mb"]
        )
        start_epoch = int(resume_checkpoint["epoch"]) + 1

        ensure_best_checkpoint(
            run_directory,
            resume_checkpoint,
            model,
            optimizer,
            config,
        )
        atomic_write_text(
            pd.DataFrame(history).to_csv(index=False),
            run_directory / "history.csv",
        )
        restore_random_state(
            resume_checkpoint["random_state"],
            train_loader.generator,
        )
        print(
            f"Resume {selected_run_id} từ epoch {start_epoch}/{EFFECTIVE_EPOCHS} "
            f"(đã hoàn thành {start_epoch - 1} epoch)."
        )
    else:
        initial_checkpoint = {
            "epoch": 0,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_accuracy": best_validation_accuracy,
            "elapsed_training_seconds": elapsed_before_session,
            "peak_gpu_memory_mb": peak_gpu_memory_before_session,
            "random_state": capture_random_state(train_loader.generator),
            "config": config,
        }
        save_checkpoint(
            initial_checkpoint,
            run_directory / "last_checkpoint.pt",
        )

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    session_training_start = time.perf_counter()
    session_epoch_seconds = []

    for epoch in range(start_epoch, EFFECTIVE_EPOCHS + 1):
        epoch_start = time.perf_counter()
        train_metrics = train_one_epoch(
            model, train_loader, criterion, optimizer, run_method, DEVICE
        )
        validation_metrics = evaluate(model, validation_loader, criterion, DEVICE)
        epoch_seconds = time.perf_counter() - epoch_start
        session_epoch_seconds.append(epoch_seconds)

        row = {
            "epoch": epoch,
            "learning_rate": scheduler.get_last_lr()[0],
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "validation_loss": validation_metrics["loss"],
            "validation_accuracy": validation_metrics["accuracy"],
            "epoch_seconds": epoch_seconds,
            "gpu_name": environment["gpu_name"],
        }
        history.append(row)
        atomic_write_text(
            pd.DataFrame(history).to_csv(index=False),
            run_directory / "history.csv",
        )

        is_new_best = validation_metrics["accuracy"] > best_validation_accuracy
        if is_new_best:
            best_epoch = epoch
            best_validation_accuracy = validation_metrics["accuracy"]

        scheduler.step()
        current_session_peak_gpu_memory_mb = (
            torch.cuda.max_memory_allocated() / (1024**2)
            if DEVICE.type == "cuda"
            else 0.0
        )
        elapsed_training_seconds = (
            elapsed_before_session + time.perf_counter() - session_training_start
        )
        peak_gpu_memory_mb = max(
            peak_gpu_memory_before_session,
            current_session_peak_gpu_memory_mb,
        )
        latest_checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "best_epoch": best_epoch,
            "best_validation_accuracy": best_validation_accuracy,
            "elapsed_training_seconds": elapsed_training_seconds,
            "peak_gpu_memory_mb": peak_gpu_memory_mb,
            "random_state": capture_random_state(train_loader.generator),
            "config": config,
        }

        # Ghi trạng thái tiếp tục trước. Nếu phiên ngắt ngay sau đó, best checkpoint
        # của epoch hiện tại có thể được dựng lại từ chính latest checkpoint này.
        save_checkpoint(
            latest_checkpoint,
            run_directory / "last_checkpoint.pt",
        )
        if is_new_best:
            save_checkpoint(
                make_best_checkpoint(
                    epoch,
                    model,
                    optimizer,
                    best_validation_accuracy,
                    config,
                ),
                run_directory / "best_checkpoint.pt",
            )

        remaining_hours = (
            float(np.mean(session_epoch_seconds)) * (EFFECTIVE_EPOCHS - epoch) / 3600
        )
        print(
            f"Epoch {epoch:03d}/{EFFECTIVE_EPOCHS} | "
            f"train loss={train_metrics['loss']:.4f}, "
            f"train acc={100 * train_metrics['accuracy']:.2f}% | "
            f"val loss={validation_metrics['loss']:.4f}, "
            f"val acc={100 * validation_metrics['accuracy']:.2f}% | "
            f"{epoch_seconds:.1f}s | còn khoảng {remaining_hours:.1f}h"
        )

    total_training_seconds = (
        elapsed_before_session + time.perf_counter() - session_training_start
    )
    current_session_peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated() / (1024**2)
        if DEVICE.type == "cuda"
        else 0.0
    )
    peak_gpu_memory_mb = max(
        peak_gpu_memory_before_session,
        current_session_peak_gpu_memory_mb,
    )

    # Load checkpoint về CPU để không tạo thêm một bản sao optimizer state trên GPU.
    checkpoint = load_checkpoint(run_directory / "best_checkpoint.pt")
    model.load_state_dict(checkpoint["model_state_dict"])
    train_checkpoint_metrics = evaluate(model, train_eval_loader, criterion, DEVICE)
    test_metrics = evaluate(model, test_loader, criterion, DEVICE)

    summary = {
        **config,
        "best_epoch": checkpoint["epoch"],
        "best_validation_accuracy": best_validation_accuracy,
        "test_accuracy": test_metrics["accuracy"],
        "test_loss": test_metrics["loss"],
        "final_train_accuracy": history[-1]["train_accuracy"],
        "final_train_loss": history[-1]["train_loss"],
        "checkpoint_train_accuracy": train_checkpoint_metrics["accuracy"],
        "generalization_gap": train_checkpoint_metrics["accuracy"] - test_metrics["accuracy"],
        "generalization_gap_definition": (
            "accuracy trên train split với nhãn sạch, không augmentation, tại best checkpoint "
            "trừ test accuracy"
        ),
        "total_training_seconds": total_training_seconds,
        "mean_epoch_seconds": float(np.mean([row["epoch_seconds"] for row in history])),
        "peak_gpu_memory_mb": peak_gpu_memory_mb,
        "gpu_name": environment["gpu_name"],
        "framework_versions": {
            "python": environment["python"],
            "torch": environment["torch"],
            "torchvision": environment["torchvision"],
        },
        "git_commit": environment["git_commit"],
        "notebook_version": NOTEBOOK_VERSION,
    }
    save_learning_curves(history, run_directory / "learning_curves.png")
    atomic_write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        run_directory / "summary.json",
    )

    missing_artifacts = REQUIRED_ARTIFACTS - {path.name for path in run_directory.iterdir()}
    if missing_artifacts:
        raise RuntimeError(f"Thiếu artifact: {sorted(missing_artifacts)}")

    return {
        "test_loader": test_loader,
        "run_directory": run_directory,
        "history": pd.DataFrame(history),
        "summary": summary,
    }


def validate_run_artifacts(artifacts, expected_method, expected_rho):
    """Kiểm tra schema artifact và load checkpoint để inference một batch."""
    run_directory = artifacts["run_directory"]
    actual_files = {path.name for path in run_directory.iterdir()}
    missing_artifacts = REQUIRED_ARTIFACTS - actual_files
    if missing_artifacts:
        raise RuntimeError(f"Thiếu artifact: {sorted(missing_artifacts)}")

    config = json.loads((run_directory / "config.json").read_text(encoding="utf-8"))
    summary = json.loads((run_directory / "summary.json").read_text(encoding="utf-8"))
    history = pd.read_csv(run_directory / "history.csv")
    missing_summary_fields = REQUIRED_SUMMARY_FIELDS - set(summary)
    missing_history_columns = REQUIRED_HISTORY_COLUMNS - set(history.columns)
    if missing_summary_fields:
        raise RuntimeError(f"summary.json thiếu field: {sorted(missing_summary_fields)}")
    if missing_history_columns:
        raise RuntimeError(f"history.csv thiếu cột: {sorted(missing_history_columns)}")
    if config["method"] != expected_method or summary["method"] != expected_method:
        raise RuntimeError("Method trong artifact không khớp với cấu hình đang chạy.")
    if config["rho"] != expected_rho or summary["rho"] != expected_rho:
        raise RuntimeError("Rho trong artifact không khớp với cấu hình đang chạy.")
    if len(history) != EFFECTIVE_EPOCHS:
        raise RuntimeError(
            f"history.csv có {len(history)} dòng, cần {EFFECTIVE_EPOCHS} dòng."
        )

    checkpoint = load_checkpoint(run_directory / "best_checkpoint.pt")
    reloaded_model = build_model(checkpoint["config"]["model"]).to(DEVICE)
    reloaded_model.load_state_dict(checkpoint["model_state_dict"])
    reloaded_model.eval()
    images, _ = next(iter(artifacts["test_loader"]))
    with torch.no_grad():
        logits = reloaded_model(images.to(DEVICE, non_blocking=True))
    if logits.shape != (images.size(0), 10):
        raise RuntimeError(f"Logits có shape không hợp lệ: {tuple(logits.shape)}")

    del reloaded_model, checkpoint, logits
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return {
        "run_id": config["run_id"],
        "method": expected_method,
        "rho": expected_rho,
        "epochs": len(history),
        "test_accuracy": summary["test_accuracy"],
        "artifact_status": "OK",
        "checkpoint_inference": "OK",
    }


def load_completed_run_artifacts(run_directory, run_method, run_id, rho):
    """Nạp run đã hoàn tất và xác minh nó khớp cấu hình hiện tại."""
    train_loader, train_eval_loader, validation_loader, test_loader, split_info = (
        build_dataloaders()
    )
    del train_loader, train_eval_loader, validation_loader

    parameter_count = sum(
        parameter.numel() for parameter in build_model(EFFECTIVE_MODEL).parameters()
    )
    current_config = build_run_config(
        run_method,
        rho,
        run_id,
        split_info,
        parameter_count,
    )
    latest_checkpoint = load_checkpoint(run_directory / "last_checkpoint.pt")
    validate_resume_config(latest_checkpoint.get("config"), current_config)
    return {
        "test_loader": test_loader,
        "run_directory": run_directory,
    }


def run_planned_runs():
    """Chạy hoặc tiếp tục SAM với từng rho trong RHO_VALUES, bỏ qua run tương thích đã hoàn tất."""
    run_reports = []
    for planned_rho, planned_run_id in PLANNED_RUN_IDS.items():
        run_directory = RESULTS_ROOT / planned_run_id
        print(f"\n===== Bắt đầu {planned_run_id} (rho={planned_rho}) =====")
        if (run_directory / "summary.json").is_file():
            print(f"{planned_run_id} đã hoàn tất; kiểm tra artifact và bỏ qua training.")
            run_artifacts = load_completed_run_artifacts(
                run_directory,
                METHOD,
                planned_run_id,
                planned_rho,
            )
        else:
            run_artifacts = run_experiment(
                method=METHOD,
                run_id=planned_run_id,
                rho=planned_rho,
            )

        run_reports.append(validate_run_artifacts(run_artifacts, METHOD, planned_rho))
        del run_artifacts
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return pd.DataFrame(run_reports)


## 9. Tự động chạy các run của `RUN_PLAN`

Khi Run all tới cell này, notebook chạy tuần tự SAM với từng giá trị trong `RHO_VALUES` (mặc định 0.01, 0.10 rồi 0.20). Với mỗi run, notebook:

1. Bỏ qua training nếu run đã có `summary.json`, sau khi xác minh config khớp notebook hiện tại.
2. Resume từ `last_checkpoint.pt` nếu run đang dở; nếu chưa có thư mục run thì bắt đầu run mới.
3. In log mỗi epoch kèm thời gian epoch và ước lượng thời gian còn lại của run đó.
4. Đánh giá test set bằng best checkpoint theo validation accuracy, kiểm tra đủ artifact, load checkpoint và inference một batch.

Nếu Colab ngắt phiên ở bất kỳ đâu, mở lại đúng notebook và **Run all**. Không đổi `RUN_PLAN`, `SEED`, `RHO_VALUES` hay recipe giữa chừng; config khác checkpoint sẽ bị từ chối để tránh trộn run.

Mỗi run `rho` có cùng chi phí với `core_sam_s<SEED>` (WRN-28-10, 200 epochs, hai forward/backward mỗi batch). Ba run cần khoảng ba lần thời gian và dung lượng Drive của một core run SAM, nên thường phải resume qua nhiều phiên Colab.

In [ ]:
# Run all sẽ thực thi cell này sau khi Google Drive và dataset đã được kiểm tra.
run_report = run_planned_runs()
display(run_report)

## 10. Xem learning curves

`run_planned_runs()` đã kiểm tra schema artifact, load checkpoint tốt nhất theo validation accuracy và inference một batch cho từng run. Cell dưới đây hiển thị learning curves của các run vừa chạy từ Google Drive.

In [ ]:
figure, axes = plt.subplots(len(PLANNED_RUN_IDS), 1, figsize=(12, 4.5 * len(PLANNED_RUN_IDS)))
axes = np.atleast_1d(axes)
for axis, (planned_rho, planned_run_id) in zip(axes, PLANNED_RUN_IDS.items()):
    curve_path = RESULTS_ROOT / planned_run_id / "learning_curves.png"
    axis.imshow(plt.imread(curve_path))
    axis.set_title(f"Learning curves - {planned_run_id} (rho={planned_rho})")
    axis.axis("off")
figure.tight_layout()
plt.show()

## 11. Load checkpoint và dự đoán một batch

Cell cuối tạo một model mới, load `best_checkpoint.pt` của run đầu tiên trong `RHO_VALUES` và dự đoán một batch test. Đây là luồng dùng lại cho demo: khi thuyết trình chỉ cần trỏ tới checkpoint của full run đã chuẩn bị trước, không chạy lại 200 epochs.

In [ ]:
@torch.no_grad()
def load_checkpoint_and_predict(checkpoint_path, loader, device=DEVICE):
    """Load model mới từ checkpoint và trả về một batch dự đoán."""
    checkpoint_path = Path(checkpoint_path)
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    checkpoint_config = checkpoint["config"]

    reloaded_model = build_model(checkpoint_config["model"]).to(device)
    reloaded_model.load_state_dict(checkpoint["model_state_dict"])
    reloaded_model.eval()

    images, targets = next(iter(loader))
    logits = reloaded_model(images.to(device, non_blocking=True))
    predictions = logits.argmax(dim=1).cpu()

    if logits.shape != (images.size(0), 10):
        raise RuntimeError(f"Logits có shape không hợp lệ: {tuple(logits.shape)}")

    return reloaded_model, images, targets, predictions


def show_predictions(images, targets, predictions, max_images=8):
    """Hiển thị một số ảnh cùng nhãn thật và dự đoán."""
    image_count = min(max_images, len(images))
    figure, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.ravel()
    mean = torch.tensor(CIFAR10_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR10_STD).view(3, 1, 1)

    for index, axis in enumerate(axes):
        if index >= image_count:
            axis.axis("off")
            continue
        image = (images[index].cpu() * std + mean).clamp(0, 1)
        image = image.permute(1, 2, 0).numpy()
        true_name = CIFAR10_CLASSES[int(targets[index])]
        predicted_name = CIFAR10_CLASSES[int(predictions[index])]
        color = "green" if true_name == predicted_name else "red"
        axis.imshow(image)
        axis.set_title(f"Thật: {true_name}\nĐoán: {predicted_name}", color=color)
        axis.axis("off")

    figure.tight_layout()
    plt.show()


demo_run_id = next(iter(PLANNED_RUN_IDS.values()))
_, _, _, demo_test_loader, _ = build_dataloaders()
reloaded_model, images, targets, predictions = load_checkpoint_and_predict(
    RESULTS_ROOT / demo_run_id / "best_checkpoint.pt",
    demo_test_loader,
)
print(f"Dự đoán bằng best checkpoint của {demo_run_id}")
show_predictions(images, targets, predictions)

## 12. Kiểm tra sau khi chạy

- [ ] Artifact nằm trong `MyDrive/CS2229-SAM/results/<run_id>/` với đủ bảy file: `config.json`, `history.csv`, `summary.json`, `best_checkpoint.pt`, `last_checkpoint.pt`, `learning_curves.png`, `environment.txt`.
- [ ] Bảng ở mục 9 có `artifact_status = OK` và `checkpoint_inference = OK` cho cả ba run trong `RHO_VALUES`.
- [ ] `history.csv` có đúng `EFFECTIVE_EPOCHS` dòng (200 với `RUN_PLAN = "rho"`).
- [ ] `summary.json` ghi đúng `run_plan`, `seed`, `method`, `rho`, `notebook_version` và `base_notebook_version`.
- [ ] Ghi lại các GPU khác nhau trong cột `gpu_name` của `history.csv` nếu run được resume trên nhiều loại GPU.

Kết quả smoke test chỉ xác nhận pipeline hoạt động. Không dùng accuracy của tập con smoke để kết luận về ảnh hưởng của `rho`, và không ngoại suy thời gian hay bộ nhớ của WRN-16-1 cho WRN-28-10.

`final_train_accuracy` dùng nhãn training sau khi thêm noise và có augmentation. `checkpoint_train_accuracy` cùng `generalization_gap` dùng train split với nhãn sạch, không augmentation, tại best checkpoint.

Notebook resume ở ranh giới epoch. Khi Colab ngắt phiên, mở lại cùng notebook, mount lại cùng tài khoản Google Drive và Run all. Không xóa thư mục run dở. Nếu cần bắt đầu lại một run từ đầu, hỏi cả nhóm trước, rồi đổi tên thư mục cũ trên Drive thay vì xóa để giữ bằng chứng.

Rho ablation chỉ có một seed nên chỉ được trình bày như kết quả khám phá, không dùng để khẳng định một giá trị `rho` tốt hơn một cách ổn định.